# BƯỚC 1 & 2: KHẢO SÁT & KIỂM TRA CHẤT LƯỢNG DỮ LIỆU THÔ (DATA QUALITY & PROFILING)

---
**Mục tiêu của Notebook:**
1. **Trích xuất dữ liệu cần thiết (Feature Selection)** từ kho dữ liệu thô Bronze (bronze_listings + bronze_details_valid).
2. **Kiểm tra cấu trúc và kiểu dữ liệu (Schema & Data Types)** của toàn bộ các trường tuyển dụng.
3. **Đo lường giá trị khuyết (Missing / Null Report)**: Thống kê số lượng Null/None và chuỗi rỗng trên từng cột.
4. **Kiểm tra trùng lặp (Duplicates Report)**: Phát hiện trùng lặp theo ID nguồn, URL bài đăng và thực thể công việc [Công ty + Chức danh].
5. Đưa ra cơ sở thực tế để thực hiện bước Tiền xử lý & Làm sạch dữ liệu tiếp theo.

In [1]:
import os
import json
import pandas as pd
import numpy as np

from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [2]:
# Duong dan toi 2 file Bronze Master (tuong thich moi thu muc lam viec)
def resolve_path(rel_subpath):
    candidates = [
        rel_subpath,
        f"../{rel_subpath}",
        f"../../{rel_subpath}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath(rel_subpath)

listings_path = resolve_path('data/bronze/bronze_listings_master.json')
details_path = resolve_path('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f'Dang doc Listings tu: {listings_path}')
print(f'Dang doc Details tu : {details_path}\n')
print(f'Bang Listings: {len(df_listings):,} dong, {df_listings.shape[1]} cot')
print(f'Bang Details : {len(df_details):,} dong, {df_details.shape[1]} cot\n')

# Thong ke so luong tin theo tung san
summary_sources = pd.DataFrame({
    'So Listings': df_listings['source'].value_counts(),
    'So Details Valid': df_details['source'].value_counts()
})
display(summary_sources)


Dang doc Listings tu: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
Dang doc Details tu : c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

Bang Listings: 5,666 dong, 20 cot
Bang Details : 5,631 dong, 35 cot



,So Listings,So Details Valid
source,,
topcv,3695,3661
vietnamworks,1786,1785
topdev,185,185


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [3]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
match_rate = len(df_merged) / len(df_details) * 100
print(f'  • Tỷ lệ khớp khóa (Match)  : {match_rate:.2f}% ({len(df_merged):,}/{len(df_details):,} tin chi tiết có listing tương ứng)')

# Ghép kiểu 'inner' sẽ BỎ các tin chi tiết không có listing ➔ liệt kê ra thay vì bỏ qua trong im lặng
key = ['source', 'source_job_id']
unmatched = df_details.merge(df_list_sub[key], on=key, how='left', indicator=True).query('_merge == "left_only"')
if len(unmatched):
    print(f'  ⚠ {len(unmatched):,} tin chi tiết KHÔNG có listing (bị loại khỏi bảng ghép), theo nguồn:',
          unmatched['source'].value_counts().to_dict())
    display(unmatched[key + ['title_raw', 'company_name_raw', 'crawled_at']].head(10))
else:
    print('  • Mọi tin chi tiết đều có listing tương ứng.')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 5,631 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (5,631/5,631 tin chi tiết có listing tương ứng)
  • Mọi tin chi tiết đều có listing tương ứng.


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [4]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

# 4. Bản ghi trong Bronze KHÔNG có dấu vết kiểm định (dq_status trống): tức là được đưa vào Bronze mà không qua QualityValidator
no_dq = df_merged[df_merged['dq_status'].isna()] if 'dq_status' in df_merged else df_merged.iloc[:0]
print(f'--- 4. BẢN GHI THIẾU THÔNG TIN KIỂM ĐỊNH (dq_status trống): {len(no_dq):,} ---')
if len(no_dq):
    display(no_dq[['source', 'source_job_id', 'title_raw', 'company_name_raw', 'location_raw', 'crawled_at']])

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 5631 entries, 0 to 5630
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             5631 non-null   str   
 1   source                5631 non-null   str   
 2   source_job_id         5631 non-null   str   
 3   job_url               5631 non-null   str   
 4   title_raw             5631 non-null   str   
 5   company_name_raw      5631 non-null   str   
 6   company_url           5631 non-null   str   
 7   company_size_raw      5631 non-null   str   
 8   company_location_raw  5631 non-null   str   
 9   industry_raw          5631 non-null   str   
 10  salary_raw            5631 non-null   str   
 11  experience_raw        5631 non-null   str   
 12  education_raw         5631 non-null   str   
 13  location_raw          5631 non-null   str   
 14  work_mode_raw         5631 non-null   str   
 15  employmen

,Kiểu Dữ Liệu,Số lượng Null/None,"Số lượng Chuỗi Rỗng ("""")",Tổng Số Khuyết,Tỷ Lệ Khuyết (%)
preferred_skills_raw,str,0,5631,5631,100.00
updated_at_raw,str,0,3846,3846,68.30
overview_raw,str,0,3515,3515,62.42
company_size_raw,str,0,1161,1161,20.62
education_raw,str,0,867,867,15.40
working_time_raw,str,0,330,330,5.86
work_mode_raw,str,0,50,50,0.89
company_logo_url,str,0,39,39,0.69
requirements_raw,str,0,27,27,0.48
required_skills_raw,str,0,5,5,0.09


--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---


,Tiêu Chí Trùng Lặp,Số Bản Ghi Trùng,Tỷ Lệ Trùng (%)
0,"Theo cặp khóa nguồn [source, source_job_id]",0,0.00
1,Theo đường dẫn [job_url],0,0.00
2,"Theo cùng vị trí & cùng công ty [title_raw, company_name...",309,5.49


--- 4. BẢN GHI THIẾU THÔNG TIN KIỂM ĐỊNH (dq_status trống): 0 ---


## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép 


In [5]:
print(f' XEM DỌC TOÀN BỘ {df_merged.shape[1]} CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:


,3135,1835,3832
detail_id,topcv_2000065,vietnamworks_2118281,topcv_2046627
source,topcv,vietnamworks,topcv
source_job_id,2000065,2118281,2046627
job_url,https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-phan-...,https://www.vietnamworks.com/ky-su-backend-2118281-jv,https://www.topcv.vn/viec-lam/truong-phong-lap-trinh/204...
title_raw,Nhân Viên Kinh Doanh Phần Mềm (Website),Kỹ Sư Backend,Trưởng Phòng Lập Trình
company_name_raw,CÔNG TY TNHH THƯƠNG MẠI DỊCH VỤ WEB IDEAS,Điện Toán Group,CÔNG TY CỔ PHẦN LIFETEX
company_url,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-dic...,https://www.vietnamworks.com/nha-tuyen-dung/đien-toan-gr...,https://www.topcv.vn/cong-ty/cong-ty-co-phan-lifetex/178...
company_size_raw,10-24 nhân viên,,25-99 nhân viên
company_location_raw,"361 Đường Phạm Văn Bạch, Phường 15, Quận Tân Bình, Thành...","437/16 Hoàng Văn Thụ, Phường Tân Sơn Nhất, Thành phố Hồ ...","Số 15 Tố Hữu, Thanh Xuân, Hà Nội"
industry_raw,"Kinh doanh / Bán hàng, IT Phần cứng / Mạng, Công nghệ th...",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,"IT phần mềm, Công nghệ thông tin"
